Посмотреть столбцы в таблицах в файле Data в директории

In [1]:
import pandas as pd
import os

for file in os.listdir("data"):
    if file.endswith(".csv"):
        df = pd.read_csv(f"data/{file}")
        print(f"\n--- {file} ---")
        print(df.shape)
        print(df.columns.tolist())


--- bureau.csv ---
(24689, 8)
['client_id', 'bureau_account_id', 'account_type', 'opened_days_ago', 'credit_limit', 'current_balance', 'max_dpd_last_12m', 'bureau_status']

--- previous_loans.csv ---
(12762, 7)
['client_id', 'previous_loan_id', 'previous_amount', 'previous_term_months', 'closed_days_ago', 'was_overdue', 'max_overdue_days']

--- test.csv ---
(2520, 24)
['application_id', 'client_id', 'region_coefficient_extended', 'loan_term_months', 'siberia_northern_score', 'hash_id', 'loan_amount', 'post_loan_collection_score', 'months_at_job', 'channel', 'education', 'internal_decision_code', 'monthly_income', 'region', 'employment_type', 'days_until_first_overdue', 'incoming_amount', 'age', 'dependents', 'interest_rate', 'marketing_segment', 'region_coefficient', 'requested_product', 'application_date']

--- train.csv ---
(6488, 25)
['application_id', 'client_id', 'employment_type', 'hash_id', 'region_coefficient_extended', 'age', 'incoming_amount', 'internal_decision_code', 'mont

In [1]:
import requests
import psycopg2

# 1. Запрос данных по API
response = requests.get('https://example.com')
data = response.json()

# 2. Подключение к PostgreSQL
conn = psycopg2.connect(
    dbname="mydb", user="user", password="password", host="localhost"
)
cursor = conn.cursor()

# 3. Вставка данных в таблицу
for item in data:
  cursor.execute(
      "INSERT INTO my_table (id, name) VALUES (%s, %s) ON CONFLICT (id) DO"
      " NOTHING",
      (item['id'], item['name']),
  )

conn.commit()
cursor.close()
conn.close()


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

Подключение к таблице, которая находится на сервере PgAdmin4

In [2]:
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

db_name = os.environ["POSTGRES_DB"]
db_user = os.environ["POSTGRES_USER"]
db_password = os.environ["POSTGRES_PASSWORD"]
db_host = os.environ["POSTGRES_HOST"]
db_port = os.environ["POSTGRES_PORT"]

connection_url = (
    f"postgresql+psycopg2://{db_user}:{db_password}"
    f"@{db_host}:{db_port}/{db_name}"
)

engine = create_engine(connection_url)



Перезапись данных из папки Data на сервер. Короткий путь загрузить их на сервер.

In [ ]:
# CSV читается через read_csv, а не read_sql
# df = pd.read_csv("data/train.csv")
# df.to_sql("train", engine, if_exists="replace", index=False)

In [2]:
# CSV читается через read_csv, а не read_sql
df = pd.read_csv("data/previous_loans.csv")
df.to_sql("previous_loans", engine, if_exists="replace", index=False)

762

In [3]:
# CSV читается через read_csv, а не read_sql
df = pd.read_csv("data/test.csv")
df.to_sql("test", engine, if_exists="replace", index=False)

520

In [4]:
# CSV читается через read_csv, а не read_sql
df = pd.read_csv("data/transactions.csv")
df.to_sql("transactions", engine, if_exists="replace", index=False)

300

Загрузить сведения из сервера в notebook 

In [4]:
df_previous_loans = pd.read_sql(
    "SELECT * FROM previous_loans_clean",
    engine
)

df_previous_loans.head()

,client_id,previous_loan_id,previous_amount,previous_term_months,closed_days_ago,was_overdue,max_overdue_days
0,2,L2_0,51911.71,3.0,986.0,1.0,3.0
1,2,L2_1,64582.12,24.0,1172.0,0.0,0.0
2,3,L3_2,27102.97,3.0,744.0,0.0,0.0
3,4,L4_3,56399.68,12.0,139.0,1.0,15.0
4,4,L4_4,73117.38,18.0,548.0,0.0,0.0


# Data preprocessing

In [ ]:
df_previous_loans.drop_duplicates()
df_previous_loans.shape

df_previous_loans.info()

round(df_previous_loans.isnull().sum()/len(df_previous_loans)*100, 2)

plt.figure(figsize=(14, 6))

sns.heatmap(df_previous_loans.isnull(), cmap='coolwarm')

plt.xticks(rotation=45, ha='right')
plt.title('Missing Values Heatmap')

plt.tight_layout()
plt.show()

### Handle missing values

### Удалим NAN в столбцах was_overdue и max_overdue_days, если в обоих на 1 строке был пропуск

df_previous_loans = df_previous_loans.dropna(subset=["was_overdue", "max_overdue_days"],how="all")
df_previous_loans.reset_index(drop=True, inplace=True)

### Если пропуск только у одного столбца

# Если просрочки не было, а дни просрочки неизвестны ставим 0
df_previous_loans.loc[(df_previous_loans["was_overdue"] == 0) & (df_previous_loans["max_overdue_days"].isna()),"max_overdue_days"] = 0

# Если просрочка была, но количество дней неизвестно удаляем строку
df_previous_loans = df_previous_loans.drop(df_previous_loans[(df_previous_loans["was_overdue"] == 1) & (df_previous_loans["max_overdue_days"].isna())].index)

# Если дней просрочки 0, а факт просрочки неизвестен ставим 0
df_previous_loans.loc[(df_previous_loans["max_overdue_days"] == 0) & (df_previous_loans["was_overdue"].isna()),"was_overdue"] = 0

# Если дней просрочки больше 0, а факт просрочки неизвестен ставим 1
df_previous_loans.loc[(df_previous_loans["max_overdue_days"] > 0) & (df_previous_loans["was_overdue"].isna()),"was_overdue"] = 1

### Заполнение пропусков numerical columns

cols = ["previous_amount", "previous_term_months", "closed_days_ago"]
for col in cols:
  # Заполнение с median группируя по client_id
  df_previous_loans[col] = df_previous_loans.groupby("client_id")[col].transform(lambda x: x.fillna(x.median()))
  # Заполнение оставшихся NaN общими median для всего столбца
  df_previous_loans[col] = df_previous_loans[col].fillna(df_previous_loans[col].median())
df_previous_loans.reset_index(drop=True, inplace=True)

По моему мнению такое заполнение максимально сохранит приближенность к реальным значениям

df_previous_loans.info()

### Определим выбросы

numerical_cols_previous_loans = [
    'previous_amount',
    'previous_term_months',
    'closed_days_ago',
    'max_overdue_days'
]

plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols_previous_loans):
    plt.subplot(2, 2, i + 1) # Adjust grid size as needed
    sns.boxplot(y=df_previous_loans[col])
    plt.title(f'Box Plot of {col}')
    plt.ylabel(col)
plt.tight_layout()
plt.show()

### Correlation matrix

numerical_previous_loans = list(set(df_previous_loans.columns) - set(
        [
            "previous_loan_id"
        ]
    )
)
corr_matrix_previous_loans = df_previous_loans[numerical_previous_loans].corr()
sns.heatmap(corr_matrix_previous_loans, annot=True,
    fmt=".2f")

 ### Агрегируем previous_loans перед добавлением в Train Test

df_previous_loans = df_previous_loans.drop(columns='previous_loan_id')

df_previous_num = df_previous_loans.groupby('client_id').agg(
    previous_loans_count=('client_id', 'size'),  # Количество предыдущих займов
    previous_amount_sum=('previous_amount', 'sum'),   # Общая сумма предыдущих займов
    previous_amount_mean=('previous_amount', 'mean'),  # Средняя сумма предыдущего займа
    previous_term_mean=('previous_term_months', 'mean'),  # Средний срок предыдущих займов
    previous_oldest_closed=('closed_days_ago', 'max'),  # Самый давно закрытый займ
    previous_latest_closed=('closed_days_ago', 'min'),  # Самый недавно закрытый займ
    previous_max_overdue=('max_overdue_days', 'max'),  # Максимальная просрочка
    previous_mean_overdue=('max_overdue_days', 'mean')  # Средняя просрочка
).reset_index()
df_previous_num

previous_overdue = df_previous_loans.groupby('client_id').agg(
    previous_overdue_count=('was_overdue', 'sum'),
    previous_has_overdue=('was_overdue', 'max')).reset_index()
previous_overdue

df_previous_features = (df_previous_num.merge(previous_overdue, on='client_id', how='left').fillna(0))
df_previous_features